# SVC Processing Pipeline — Interactive Tutorial

This notebook runs the **whole pipeline** on SVC HR-1024i `.sig` scans and draws a
plot at every step. It assumes that you already have a folder of raw `.sig` files.
You only need to tell the settings cell where that folder is.

| Part | What it does |
|---|---|
| 1 — Full folder | Load → inspect raw → process → visualize each step -> filter references & outliers → process → save |
| 2 — Pairs (by position) | Average consecutive scans → plot individuals + means |
| 3 — Groups (by scan number) | Average by the scan number in each filename (real-data way) |

Every code cell is explained in the text or comments just above it. Run the cells
top to bottom with **Shift + Enter**.

## Instrument overview

The SVC HR-1024i is a field spectroradiometer that records reflectance across the
visible, near-infrared, and shortwave-infrared regions using **three detector
arrays**:

| Detector array | Region | Approximate range |
|---|---:|---:|
| Silicon | VNIR | 340–1012 nm |
| InGaAs | SWIR-1 | 972–1910 nm |
| Extended InGaAs | SWIR-2 | 1894–2517 nm |

Because the arrays overlap, each raw `.sig` file contains three sequential sensor
segments with small discontinuities that must be trimmed, matched, smoothed, and
resampled into one continuous curve from 400–2500 nm.

## 1. Setup

Use a Python 3.11 or newer notebook kernel. The first code cell checks the kernel
version so pip does not fail with a confusing compatibility message, then the next
cell installs `svc-processing` and the plotting dependencies into the current kernel.
No repository clone or manual Python-path setup is required.

The public repository intentionally does **not** include raw field `.sig` files because
instrument headers can contain GPS and time metadata. Copy your authorized scans to a
folder you control, then use the containing folder, not an individual file, as
`DATA_FOLDER` below.

In [ ]:
import sys

if sys.version_info < (3, 11):
    version = ".".join(map(str, sys.version_info[:3]))
    raise RuntimeError(
        "This notebook requires Python 3.11 or newer. "
        f"The active kernel is Python {version}. "
        "Switch Jupyter to a Python 3.11+ kernel, then rerun this cell."
    )

print(f"Python {sys.version.split()[0]} kernel is ready.")

In [ ]:
%pip install --upgrade "svc-processing[demo]>=0.1.6"

In [ ]:
import os
from pathlib import Path

try:
    from pipeline.notebook import (
        build_config,            # turn a few settings into a pipeline config
        SpectraCollection,       # a folder of scans
        save_spectra_csv,        # write processed spectra to CSV
        average_pairs,           # Part 3: average scans by position
        plot_paired_averages,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import pipeline.notebook. Run the setup/install cell above in this "
        "same kernel, then rerun this import cell."
    ) from exc

print("Helpers imported from the installed svc-processing package.")

### Your settings — edit these

Set `DATA_FOLDER` to the folder containing your raw `.sig` files. Absolute paths are
simplest; examples are `Path("/Users/name/data/run_01")` on macOS/Linux and
`Path(r"C:\Users\name\data\run_01")` on Windows. A relative path such as
`Path("data/run_01")` is resolved from the folder where Jupyter is running.

Change `OUTPUT_FOLDER` if you want results somewhere else. Leave
`INSTRUMENT = "auto"` to read the instrument from the file headers, or set it
explicitly to `"bronze"` or `"silver"`.

In [ ]:
WORK_FOLDER = Path.cwd()

# REQUIRED: replace this with the folder containing your raw .sig files.
# SVC_DATA_FOLDER is optional and is mainly useful for automated/headless runs.
DATA_FOLDER = Path(os.environ.get("SVC_DATA_FOLDER", "path/to/demo/data")).expanduser()

# Results are written beneath the folder where this notebook is running.
OUTPUT_FOLDER = WORK_FOLDER / "pipeline_outputs/notebook_run"

# "auto" detects the instrument from the file headers; or use "bronze" / "silver".
INSTRUMENT = "auto"

### Stage 1 end line (where each scan gets trimmed)

`END_LINE` is the wavelength where **Stage 1** cuts off each `.sig` file. Determine it
after instrument calibration or when introducing a new SVC instrument: below `data=`,
find the maximum value in the **first (wavelength) column**, check that value across
several files, and preserve the exact decimal string.

Leave `END_LINE = None` when the installed calibrated default matches your instrument
(**2520.4 nm** for bronze, **2517.9 nm** for silver). Otherwise enter the maximum as a
string, such as `END_LINE = "2517.9"`.

In [ ]:
# Stage 1 truncation wavelength.
#   None      -> use the instrument's calibrated default (bronze 2520.4, silver 2517.9)
#   "2517.9"  -> trim every scan at this wavelength instead
# A custom value must exactly match a wavelength in the first data column; the nearest
# value is not used. A mismatch leaves trailing rows untrimmed and may cause a later
# warning about finding more than the HR-1024i's three sensor sweeps.
END_LINE = 2518.0

### Check the input folder

Before building the configuration, confirm that `DATA_FOLDER` exists and contains
`.sig` files directly inside it. Keep scans from different instruments in separate
folders.

In [ ]:
sig_files = sorted(DATA_FOLDER.glob("*.sig"))
if not sig_files:
    raise FileNotFoundError(
        f"No .sig files found in {DATA_FOLDER}. Edit DATA_FOLDER in the settings cell "
        "to point at the folder containing your scans."
    )
print(f"Found {len(sig_files)} .sig files in {DATA_FOLDER}")
print("First file:", sig_files[0].name)

### Build the config

`build_config()` bundles those settings, detects which instrument took the scans, and
fills in the parity-verified Stage 2 parameters. Printing `config` shows what it
resolved.

In [ ]:
config = build_config(
    data_folder=DATA_FOLDER,
    output_folder=OUTPUT_FOLDER,
    instrument=INSTRUMENT,
    end_line=END_LINE,
)
config            # show a friendly summary: instrument, end line, paths, processing params

### Prepare the files — Stage 1

`config.prepare()` truncates each raw `.sig` file at the end wavelength shown in the
config summary above (`END_LINE`, or the instrument default) and writes the result into
the processed folder. This is **Stage 1** of the pipeline; every step after this reads
the truncated files.

If `prepare()` warns that the end line wasn't found in your files, set `END_LINE` in the
settings cell and re-run from there.

In [ ]:
config.prepare()

---
## Part 1 — processing a folder

Two filters first remove scans that should
not be analysed:
- **Reference panels** — the white Spectralon target (reflectance ≈ 1.0 everywhere).
- **Outliers** — scans whose mean reflectance is far from the group (e.g. obstructed
  view, instrument not settled).

In [ ]:
# Load every processed scan; the collection honours the config's Stage 2 settings.
collection = SpectraCollection.from_config(config)
print(collection)

### Raw — all scans

The raw plot shows the sensor fold-backs for every scan. Reference panels appear as
nearly flat lines up near reflectance 1.0.

In [ ]:
collection.plot_raw()

### Filter

Remove the reference panels first, then the outliers (so the panels don't skew the
outlier statistics). Each call prints how many scans it removed.

In [ ]:
collection.filter_reference_scans()    # drop white-reference panels
collection.filter_outliers()           # drop scans far from the group mean

### Process every scan

In [ ]:
collection.process()                   # Stage 2 on every remaining scan
print(collection)

### Check one scan

A three-panel before/after on one scan confirms the pipeline ran correctly.

In [ ]:
collection.plot_processing_steps(spectrum_index=0)

### All cleaned spectra

In [ ]:
collection.plot()                      # every cleaned scan overlaid

### Save

Write the cleaned spectra to CSV: one row per scan, wavelength columns 400–2500 nm.
We keep the path in `spectra_csv` for the grouping steps below.

In [ ]:
spectra_csv = save_spectra_csv(collection, config.output_folder / "spectra.csv")
print("Saved:", spectra_csv)

---
## Part 3 — averaging repeat scans (by position)

Field work often takes several scans per sample. The simplest way to average them is
by **position** in the list — e.g. scans 0 and 1 are one sample, 2 and 3 the next.
Quick and intuitive; see Part 4 for the robust, real-data approach.

### Which position is which file?

In [ ]:
# Position (index) -> filename, so you can choose groups deliberately.
for i, s in enumerate(collection.spectra):
    print(f"[{i}] {s.name}")

### Define pairs and average

Each tuple contains **0-based positions** in the filtered collection. The example
below makes consecutive pairs dynamically, so it remains valid if the number of scans
changes. Edit the tuples when your measurement design uses different groupings.

In [ ]:
# Pair consecutive positions: (0, 1), (2, 3), ...
groups = [
    tuple(range(i, min(i + 2, len(collection.spectra))))
    for i in range(0, len(collection.spectra), 2)
]

print("Groups to average:", groups)
pairs = average_pairs(collection, groups=groups)
print(pairs)

Individual scans are drawn faded behind their bold group mean — one colour per
pair — so you can see both within-pair agreement and between-pair variation.

In [ ]:
plot_paired_averages(collection, pairs, groups=groups)

### Save the pairs

In [ ]:
# One row per pair, wavelength columns 400-2500 nm.
paired_csv = config.output_folder / "spectra_paired.csv"
pairs.to_csv(paired_csv)
print("Saved:", paired_csv)